# 06 · Gráficos con ggplot2

Retomaremos las mismas preguntas y archivos de la clase 05. Ahora construiremos figuras por capas: datos, correspondencias visuales y geometrías.

## Objetivos y recorrido

- Comprender `ggplot()`, `aes()` y las capas agregadas con `+`.
- Distinguir un color fijo de un color asociado a una variable.
- Representar relaciones, distribuciones, conteos y series temporales.
- Comparar grupos con facetas y exportar una figura.

Ejecuta en orden; no necesitas haber ejecutado otro notebook en esta sesión.


## Preparar ggplot2

`library(ggplot2)` carga el paquete en la sesión. Si no está instalado, ejecuta una vez `install.packages("ggplot2", repos = "https://cloud.r-project.org")` en una celda aparte; necesita conexión y permisos de instalación. No se requiere cargar todo `tidyverse`.


In [ ]:
# install.packages("ggplot2", repos = "https://cloud.r-project.org")

In [ ]:
library(ggplot2)


## Recuperar las tablas del notebook 04

Este notebook es independiente: vuelve a leer los tres CSV. Desde `estudiantes/clases/` usa `../../data/raw`; desde la raíz del repositorio cambia la carpeta a `data/raw`. En Colab carga los tres archivos y usa `/content`. Antes de continuar, las tres comprobaciones deben dar `TRUE`.

Conservamos el filtro de Colombia por `place`, los metadatos y ausentes de NASA, y las fechas calculadas con año y DOY. Las fuentes y sus enlaces están en [la clase 04](04_lectura_y_analisis_exploratorio.ipynb).


In [ ]:
carpeta_datos <- "../../data/raw"
archivo_sismos <- file.path(carpeta_datos, "Earthquakes in Colombia (from 2000).csv")
archivo_meteo <- file.path(carpeta_datos, "Datos hidrometeorológicos Medellín.csv")
archivo_pinguinos <- file.path(carpeta_datos, "DB Pinguinos.csv")
file.exists(c(archivo_sismos, archivo_meteo, archivo_pinguinos))


In [ ]:
sismos <- read.csv(archivo_sismos, fileEncoding = "UTF-8",
                   na.strings = c("", "NA"), stringsAsFactors = FALSE)
sismos <- sismos[!is.na(sismos$place) & grepl("Colombia", sismos$place, fixed = TRUE), ]
meteo <- read.csv(archivo_meteo, skip = 12, fileEncoding = "UTF-8",
                  na.strings = c("", "NA", "-999"), stringsAsFactors = FALSE)
meteo$fecha <- as.Date(paste0(meteo$YEAR, "-01-01")) + meteo$DOY - 1
meteo$mes <- as.integer(format(meteo$fecha, "%m"))
meteo <- meteo[order(meteo$fecha), ]
pinguinos <- read.csv(archivo_pinguinos, fileEncoding = "UTF-8",
                      na.strings = c("", "NA"), stringsAsFactors = FALSE)
c(sismos = nrow(sismos), dias = nrow(meteo), pinguinos = nrow(pinguinos))


### Conservar los datos que necesita cada figura

`complete.cases()` identifica filas conocidas en las columnas seleccionadas. Preparamos una tabla para la relación profundidad–magnitud y otra para al menos dos medidas corporales y la especie. No eliminamos una fila por ausentes en variables ajenas a la pregunta. Las tablas originales permanecen disponibles.


In [ ]:
pares_sismos <- sismos[complete.cases(sismos[, c("depth", "mag")]), ]
pares_bio <- pinguinos[complete.cases(pinguinos[, c("species", "flipper_length_mm", "body_mass_g")]), ]
c(pares_sismos = nrow(pares_sismos), pares_bio = nrow(pares_bio))


## Estructura básica por capas

```r
ggplot(data = tabla, mapping = aes(x = variable_x, y = variable_y)) +
  geom_point() +
  labs(title = "Título", x = "Variable X", y = "Variable Y")
```

| Componente | Función |
|---|---|
| `data` | La tabla de partida. |
| `aes()` | Asocia columnas con posiciones, colores, tamaños u otras propiedades. |
| `geom_*()` | Decide cómo se representan las observaciones. |
| `labs()` | Escribe títulos, unidades y etiquetas de leyenda. |
| `theme_*()` | Ajusta la presentación. |

Dentro de `aes()` se usan los nombres de columnas, sin `tabla$`. Escribe `+` al final de la línea cuando continúas la figura en otra línea.


In [ ]:
ggplot(pares_sismos, aes(x = depth, y = mag)) +
  geom_point()


### Guardar un gráfico como objeto y añadir detalles

Un objeto gráfico permite reutilizar el diseño. El color fijo, el tamaño y la transparencia se escriben fuera de `aes()`. `alpha` va de 0 (transparente) a 1 (opaco).


In [ ]:
figura_sismos <- ggplot(pares_sismos, aes(x = depth, y = mag)) +
  geom_point(color = "steelblue", size = 1.4, alpha = 0.35) +
  labs(title = "Profundidad y magnitud", subtitle = "Eventos con Colombia en place",
       x = "Profundidad (km)", y = "Magnitud") +
  theme_minimal()
figura_sismos


## Color asociado a una variable

`aes(color = species)` crea un color por especie y su leyenda. En cambio, `color = "steelblue"` fija el mismo color para todos los puntos. Escribir `aes(color = "steelblue")` crearía una categoría llamada “steelblue”; no fija el color del dibujo.


In [ ]:
figura_bio <- ggplot(pares_bio, aes(x = flipper_length_mm, y = body_mass_g, color = species)) +
  geom_point(alpha = 0.7) +
  labs(title = "Aleta y masa corporal", x = "Longitud de la aleta (mm)",
       y = "Masa (g)", color = "Especie") +
  theme_minimal()
figura_bio


## Facetas: un panel por grupo

`facet_wrap(~ species)` separa la figura en paneles con escalas comunes por defecto. Es útil para observar si la relación dentro de cada especie se parece a la relación de la muestra completa. No concluyas causalidad a partir de la nube.


In [ ]:
figura_bio + facet_wrap(~ species)


## Histogramas y densidades

Para una sola variable numérica basta `aes(x = variable)`. `geom_histogram()` cuenta observaciones por intervalos. Con `bins` o `binwidth` controlamos la partición; justifica el ancho en función de la variable.


In [ ]:
ggplot(pares_sismos, aes(x = mag)) +
  geom_histogram(binwidth = 0.2, boundary = 0, fill = "wheat", color = "white") +
  labs(title = "Magnitud de los eventos", x = "Magnitud", y = "Número de eventos") +
  theme_minimal()


Para superponer una densidad, ambas capas deben compartir la escala vertical de densidad. `after_stat(density)` utiliza la densidad calculada por el histograma. La curva suavizada no representa nuevos datos y puede extenderse fuera del rango físico de una variable.


In [ ]:
ggplot(pares_sismos, aes(x = mag)) +
  geom_histogram(aes(y = after_stat(density)), binwidth = 0.2, fill = "wheat", color = "white") +
  geom_density(color = "firebrick", linewidth = 1) +
  labs(title = "Distribución de magnitudes", x = "Magnitud", y = "Densidad") +
  theme_minimal()


## Boxplots con observaciones individuales

`geom_boxplot()` compara distribuciones por especie. `geom_jitter()` desplaza ligeramente los puntos para reducir superposición; el desplazamiento horizontal no es una nueva variable. `outlier.shape = NA` evita duplicar los extremos, porque todos los individuos se muestran con la segunda capa.

`fill` colorea el interior de la caja; `color` colorearía su contorno. Una caja no muestra por sí sola cuántas observaciones hay; por eso añadimos los puntos.


In [ ]:
set.seed(2026)
figura_masa <- ggplot(pares_bio, aes(x = species, y = body_mass_g, fill = species)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.5) +
  geom_jitter(width = 0.12, height = 0, alpha = 0.35, size = 1) +
  labs(title = "Distribución de masa por especie", x = "Especie", y = "Masa (g)") +
  guides(fill = "none") +
  theme_minimal()
figura_masa


## Barras: contar filas o usar un resumen existente

`geom_bar()` cuenta filas para cada categoría. `geom_col()` usa una altura que ya está en la tabla, por ejemplo una media calculada antes. Confundirlos puede cambiar lo que representa el eje vertical.


In [ ]:
ggplot(pinguinos, aes(x = species)) +
  geom_bar(fill = "steelblue") +
  labs(title = "Composición de la muestra", x = "Especie", y = "Individuos registrados") +
  theme_minimal()


In [ ]:
masa_media <- aggregate(body_mass_g ~ species, data = pares_bio, FUN = mean)
ggplot(masa_media, aes(x = species, y = body_mass_g)) +
  geom_col(fill = "steelblue") +
  labs(title = "Masa media por especie", x = "Especie", y = "Masa media (g)") +
  theme_minimal()


La media en barras oculta dispersión y tamaño de muestra. Compárala con el boxplot: pueden tener centros parecidos y distribuciones diferentes. No se han calculado intervalos de confianza en esta figura.

## Serie temporal: el mismo mes entre años

Repetimos el resumen de enero. `geom_line()` conecta según el eje horizontal y `geom_point()` marca cada promedio. Son promedios mensuales de temperatura de la celda NASA POWER; la figura no demuestra por sí sola una tendencia climática.


In [ ]:
enero <- meteo[meteo$mes == 1 & !is.na(meteo$T2M), ]
table(enero$YEAR)
temperatura_enero <- aggregate(T2M ~ YEAR, data = enero, FUN = mean)
ggplot(temperatura_enero, aes(x = YEAR, y = T2M)) +
  geom_line(color = "tomato3") +
  geom_point(color = "tomato3") +
  labs(title = "Temperatura media de enero", x = "Año", y = "Temperatura a 2 m (°C)") +
  theme_minimal()


### Precipitación diaria: segmentos desde cero

Cada segmento termina en la precipitación de un día. Conservamos un año completo para no comparar periodos de distinta duración.


In [ ]:
meteo_2025 <- meteo[meteo$YEAR == 2025 & !is.na(meteo$PRECTOTCORR), ]
ggplot(meteo_2025, aes(x = fecha, y = PRECTOTCORR)) +
  geom_segment(aes(xend = fecha, yend = 0), color = "steelblue") +
  labs(title = "Precipitación diaria · 2025", x = "Fecha", y = "Precipitación (mm/día)") +
  theme_minimal()


## Zoom sin retirar datos del cálculo

`coord_cartesian()` limita la ventana visible y conserva los datos usados por las capas estadísticas. Limitar la escala con `xlim()` o `ylim()` puede eliminar observaciones antes del cálculo. Al presentar un zoom, indica que estás mostrando una parte del rango.


In [ ]:
figura_sismos + coord_cartesian(xlim = c(0, 200)) + labs(subtitle = "Zoom: profundidades de 0 a 200 km")


## Guardar la figura

`ggsave()` recibe el objeto, el nombre y el tamaño. La extensión indica el formato: PNG para imagen raster, PDF para una salida vectorial. Explicitar `plot` evita guardar por accidente la última figura mostrada.


In [ ]:
dir.create("figuras", showWarnings = FALSE)
ggsave("figuras/masa_por_especie_ggplot.png", plot = figura_masa,
       width = 8, height = 5, units = "in", dpi = 200)


## Documentación para consultar

La documentación de [ggplot()](https://ggplot2.tidyverse.org/reference/ggplot.html) desarrolla la estructura por capas. Puedes abrir `?geom_point`, `?geom_boxplot` y `?facet_wrap` desde R.

